# Set up a run and build the HPC zip

This is the notebook that decides **what to run**. It asks which dataset, which
feature sections, which evaluation methods and which GP settings, writes the run
config, and packs a zip to upload.

Nothing is fitted here. The zip carries the engine, that dataset's prepared
inputs and the submit scripts, and nothing else -- a few hundred KB rather than
the whole repository.

The order is: **1** see what is available, **2** edit `CHOICES`, **3** (optional)
adjust with the widgets, **4** preview, **5** write the config, **6** build the
zip.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()          # this notebook lives in gp_collab_hub/
sys.path.insert(0, str(ROOT))

from gpc import bundle as bundle_mod
from gpc.config import (FIVE, TWO, bundle_group, bundle_name, bundle_target,
                        read_json, validate_config, write_json, write_yaml)
from gpc.data import load_bundle
from gpc.splits import fold_table

pd.set_option("display.width", 200, "display.max_columns", 60)

# Every dataset with a prepared bundle. To add one, copy datasets/_template/
# and run its intake notebook; it shows up here with no other change.
available = []
for directory in sorted((ROOT / "datasets").glob("*/inputs/config.json")):
    prepared = read_json(directory)
    available.append({
        "dataset": directory.parent.parent.name,
        "name": bundle_name(prepared),
        "group": bundle_group(prepared),
        "target": bundle_target(prepared),
        "rows": prepared["data"]["expected_rows"],
        "groups": prepared["data"].get("expected_groups",
                                       prepared["data"].get("expected_ligands")),
        "conditions": len(prepared["data"]["common_categorical"]),
        "methods": ", ".join(prepared["evaluation"]["methods"]),
    })
available = pd.DataFrame(available)
display(available)

## 1. The choices

Edit this cell. It is the authority on what gets run -- the widget panel below
only writes back into this same dictionary, so the notebook stays re-runnable
top to bottom whether or not `ipywidgets` is installed.

**Feature sections.** `group_ohe` is the group identity one-hot (the baseline
that carries no chemistry), `selected_2` and `selected_5` are the short
descriptor sets, `pc_top` the top-loaded original descriptors and `pc_scores`
the four reference-PCA components.

**Methods.** `lolo` holds out one ligand/catalyst at a time -- the
out-of-distribution question. `iid_stratified_<n>` is its matched control and
**n must equal the group count**, or the two stop being comparable; the builder
refuses a mismatch. `kfold_stratified_5` is ordinary CV and `kfold_5` is the
same without stratification.

**`feature_columns`** is the "different features" escape hatch: redefine which
descriptor columns `selected_2` / `selected_5` / `pc_top` feed. Leave it empty
to keep the shared definitions, which is what makes sections comparable *across*
datasets. Any override is recorded in the run config and in every task's
meta.json, so a run is never silently non-standard.

In [ ]:
CHOICES = {
    # ---- what to run it on -------------------------------------------------
    "dataset": "perera",              # a row of the table above

    # ---- feature sections --------------------------------------------------
    "models": ["group_ohe", "selected_2", "selected_5", "pc_top", "pc_scores"],

    # ---- evaluation methods ------------------------------------------------
    # None -> take the bundle's own list, which already has the right fold
    # count for its group count. Set a list to override.
    "methods": None,

    # ---- redefine a section's descriptors (optional) -----------------------
    # e.g. {"selected_2": ["vbur_pct_boltz", "vbur_pct_delta"]}
    "feature_columns": {},

    # ---- GP hyperparameters ------------------------------------------------
    "gp": {
        "kernel": "RBF",              # RBF, Matern32, Matern52, Tanimoto*
        "grouping": "all",            # all | group_conditions | per_field
        "mixing_method": "product",
        "ard": True,                  # per-column lengthscales; needed for
                                      # feature importance in the review
        "dtype": "float32",
        "noise": None,                # None -> learned, floor below
        "noise_floor": 0.01,
        "outputscale": None,          # None -> learned
        "train_jitter": 1e-4,
        "predict_jitter": 1e-4,
        "restarts": 1,
        "n_epochs": 400,
        "learning_rate": 0.01,
        "prior": True,
        "normalize_y": True,
        "use_cuda": True,
    },
    "seed": 42,
    "stratify": True,
    "threads": 4,

    # ---- per-section GP overrides (optional) -------------------------------
    # {"pc_scores": {"n_epochs": 1600, "restarts": 3, "walltime": "11:59:00"}}
    "model_overrides": {},

    # ---- the cluster -------------------------------------------------------
    "hpc": {
        "conda_env": "/usr/local/usrapps/ddomlab/kagoble/gp_collab_hub_py312",
        "walltime": "03:59:00",       # per task, GPU
        "cpu_walltime": "05:59:00",   # per task, CPU fallback
        "gpu_request": "gpu:l40:1",
        "mem": "16G",
        "cpus_per_task": 4,           # keep in step with threads above
        "pilot": 0,                   # 1 -> 10 epochs, to size walltime first
        "submit_collection": 1,       # chain `gpc collect` behind the jobs
    },

    # ---- bookkeeping -------------------------------------------------------
    "run_tag": None,                  # None -> <dataset>_<today>
    "config_format": "both",          # json | yaml | both
}

## 2. Optional: pick with widgets instead

Needs `ipywidgets`. Anything changed here is written straight back into
`CHOICES`, so it is equivalent to editing the cell above -- and skipping this
cell entirely costs nothing.

In [ ]:
try:
    import ipywidgets as W
except ImportError:
    print("ipywidgets not installed -- edit CHOICES above instead.")
else:
    ALL_MODELS = ["group_ohe", "selected_2", "selected_5", "pc_top", "pc_scores",
                  "pc_scores_long"]
    prepared = read_json(ROOT / "datasets" / CHOICES["dataset"] / "inputs" / "config.json")

    dataset_w = W.Dropdown(options=list(available.dataset), value=CHOICES["dataset"],
                           description="Dataset:")
    models_w = W.SelectMultiple(options=ALL_MODELS, value=tuple(CHOICES["models"]),
                                rows=6, description="Sections:")
    methods_w = W.SelectMultiple(options=prepared["evaluation"]["methods"],
                                 value=tuple(CHOICES["methods"]
                                             or prepared["evaluation"]["methods"]),
                                 rows=4, description="Methods:")
    ard_w = W.Checkbox(value=CHOICES["gp"]["ard"], description="ARD lengthscales")
    epochs_w = W.IntText(value=CHOICES["gp"]["n_epochs"], description="Epochs:")
    kernel_w = W.Dropdown(options=["RBF", "Matern32", "Matern52"],
                          value=CHOICES["gp"]["kernel"], description="Kernel:")
    walltime_w = W.Text(value=CHOICES["hpc"]["walltime"], description="Walltime:")
    gpu_w = W.Text(value=CHOICES["hpc"]["gpu_request"], description="GPU:")
    mem_w = W.Text(value=CHOICES["hpc"]["mem"], description="Memory:")
    pilot_w = W.Checkbox(value=bool(CHOICES["hpc"]["pilot"]),
                         description="Pilot (10 epochs)")

    def _sync(_=None):
        CHOICES["dataset"] = dataset_w.value
        CHOICES["models"] = list(models_w.value)
        CHOICES["methods"] = list(methods_w.value)
        CHOICES["gp"]["ard"] = ard_w.value
        CHOICES["gp"]["n_epochs"] = int(epochs_w.value)
        CHOICES["gp"]["kernel"] = kernel_w.value
        CHOICES["hpc"]["walltime"] = walltime_w.value
        CHOICES["hpc"]["gpu_request"] = gpu_w.value
        CHOICES["hpc"]["mem"] = mem_w.value
        CHOICES["hpc"]["pilot"] = int(pilot_w.value)

    def _on_dataset(change):
        # A dataset's methods carry its own fold count, so the list has to
        # follow the dataset rather than persist across a change of it.
        cfg = read_json(ROOT / "datasets" / change["new"] / "inputs" / "config.json")
        methods_w.options = cfg["evaluation"]["methods"]
        methods_w.value = tuple(cfg["evaluation"]["methods"])
        _sync()

    dataset_w.observe(_on_dataset, names="value")
    for widget in (dataset_w, models_w, methods_w, ard_w, epochs_w, kernel_w,
                   walltime_w, gpu_w, mem_w, pilot_w):
        widget.observe(_sync, names="value")

    display(W.VBox([
        W.HTML("<b>What to run</b>"), dataset_w, models_w, methods_w,
        W.HTML("<b>Model</b>"), kernel_w, ard_w, epochs_w,
        W.HTML("<b>Cluster</b>"), walltime_w, gpu_w, mem_w, pilot_w,
    ]))

## 3. Preview

What this will actually submit: the task registry, the folds each method makes,
and the total GPU time being asked for. Read the fold table before submitting --
it is the cheapest place to notice that a method is not doing what you meant.

In [ ]:
DATASET_DIR = ROOT / "datasets" / CHOICES["dataset"] / "inputs"
reactions, ligands, reference, prepared = load_bundle(DATASET_DIR)
group = bundle_group(prepared)
methods = CHOICES["methods"] or list(prepared["evaluation"]["methods"])

run_config = {
    "schema_version": 2,
    "dataset": CHOICES["dataset"],
    "run_tag": CHOICES["run_tag"],
    "paths": {"bundle": f"datasets/{CHOICES['dataset']}/inputs", "runs": "runs/local"},
    "run": {
        "models": list(CHOICES["models"]),
        "methods": list(methods),
        "seed": CHOICES["seed"],
        "stratify": CHOICES["stratify"],
        "threads": CHOICES["threads"],
        "model_overrides": dict(CHOICES["model_overrides"]),
        "feature_columns": dict(CHOICES["feature_columns"]),
    },
    "gp": dict(CHOICES["gp"]),
    "hpc": dict(CHOICES["hpc"]),
}
validate_config(run_config)                 # typos fail here, not on the cluster
bundle_mod.check_methods(methods, prepared)  # and a mismatched control fails here

tag = bundle_mod.run_tag(run_config)
n_tasks = len(run_config["run"]["models"]) * len(run_config["run"]["methods"])
hours = sum(int(h) + int(m) / 60 for h, m, _ in
            [bundle_mod.hpc_block(run_config)["walltime"].split(":")]) * n_tasks

print(f"dataset   {bundle_name(prepared)}  ({len(reactions)} rows, "
      f"{reactions[group].nunique()} {group}s, target {bundle_target(prepared)})")
print(f"run tag   {tag}")
print(f"tasks     {n_tasks} = {len(run_config['run']['models'])} sections "
      f"x {len(run_config['run']['methods'])} methods")
print(f"budget    up to {hours:.1f} GPU-hours at "
      f"{bundle_mod.hpc_block(run_config)['walltime']} per task")
if run_config["run"]["feature_columns"]:
    print(f"OVERRIDE  {run_config['run']['feature_columns']} "
          f"-- this run is not comparable to a standard one")

display(pd.concat([fold_table(reactions, m, CHOICES["seed"], CHOICES["stratify"], group)
                   for m in methods], ignore_index=True))

## 4. Write the run config

JSON is the authority: the zip always carries JSON so the cluster never needs
PyYAML to start a job. The YAML copy is for hand editing between runs -- point
`gpc --config` at either.

In [ ]:
CONFIG_DIR = ROOT / "configs"
written = []

if CHOICES["config_format"] in ("json", "both"):
    path = CONFIG_DIR / f"{tag}.json"
    write_json(path, run_config)
    written.append(path)

if CHOICES["config_format"] in ("yaml", "both"):
    try:
        path = CONFIG_DIR / f"{tag}.yaml"
        write_yaml(path, run_config)
        written.append(path)
    except ImportError:
        print("PyYAML not installed, so no .yaml copy was written "
              "(`pip install pyyaml` if you want one). The JSON is complete.")

for path in written:
    print("wrote", path.relative_to(ROOT))

## 5. Build the zip

Upload the result, unzip it, and follow `RUN_ON_HPC.md` inside. The run tag,
conda prefix, walltime, GPU type and memory are already written into the submit
scripts, so nothing needs editing on the cluster.

`full_inputs=True` also ships the bundle's provenance tables and the whole
1,223-ligand Kraken reference. Training does not read either -- only turn it on
if you want to regenerate figures on the cluster rather than locally.

In [ ]:
zip_path = bundle_mod.write_zip(run_config,
                                config_path=written[0] if written else None,
                                full_inputs=False)

import zipfile
with zipfile.ZipFile(zip_path) as archive:
    contents = pd.DataFrame(
        [{"file": i.filename.split("/", 1)[1], "KB": round(i.file_size / 1024, 1)}
         for i in archive.infolist()])
display(contents.sort_values("KB", ascending=False).head(12))
print(f"\n{len(contents)} files, {zip_path.stat().st_size / 1024:,.0f} KB total")
print(f"\nUpload: {zip_path}")

## 6. When the run comes back

Bring back the whole `runs/<run tag>/` folder, then open **`make_report.ipynb`**
(or run the one-liner it wraps) to get `results/<dataset>_<date>/` with the
figures and tables.